In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

pd.set_option('display.max_columns', 100)

df = pd.read_csv("C:/Users/user/Desktop/Python_Data_Project/risk_credit_card_analysis/data/processed/cleaned_data.csv", index_col=[0])

## Feature engineering starts here:

In [61]:
# Making utilization band / rate columns:

df['utilization_rate'] = round((df['bill_sep'] / df['limit_bal'])*100,2)
df['utilization_band'] = pd.cut(df['utilization_rate'],
                                bins=[-np.inf, 0, 30, 60, 90, np.inf],
                                right=False,
                                labels=['No Balance', 'Under 30%', '30-59%', '60-89%', '90%+'])


# Number of late payment (in months) per customer:
MONTHS = ['sep', 'aug', 'jul', 'jun', 'may', 'apr']
PAY_COLS = [f'pay_{x}' for x in MONTHS]
df['months_late'] = (df[PAY_COLS] >=1).sum(axis=1)


# Latest payment ratio:
pay_ratio = df["paid_sep"] / df["bill_aug"].where(df["bill_aug"] > 0)
df["payment_band"] = np.select(
    [pay_ratio.isna(), pay_ratio < 0.05, pay_ratio < 0.25, pay_ratio < 1],
    ["0. No balance due", "1. Under 5% (minimum or less)", "2. 5-24%", "3. 25-99%"],
    default="4. Paid in full"
)

# Select all wanted columns for machine learning:

df_clean = df[['limit_bal', 'sex', 'education', 'marital_status',
               'age_band', 'utilization_rate', 'months_late',
               'payment_band', 'is_default']].copy()

In [62]:
# Normalizing values for VIF Test:

df_clean['sex'] = df_clean['sex'].map({
    'Male':1,
    'Female':0
})
df_clean['education'] = df_clean['education'].map({
    'University':2,
    'Graduate school':1,
    'High school':3,
    'Others/Unknown':4
})
df_clean['marital_status'] = df_clean['marital_status'].map({
    'Married':1,
    'Single':2,
    'Others/Unknown':3
})
df_clean['age_band'] = df_clean['age_band'].map({
    '<25':1,
    '25-34':2,
    '35-44':3,
    '45-54':4,
    '55+':5
})
df_clean['payment_band'] = df_clean['payment_band'].map({
    '1. Under 5% (minimum or less)':1,
    '2. 5-24%':2,
    '3. 25-99%':3,
    '4. Paid in full':4,
    '0. No balance due':0
})

### Checking the assumptions for logistic regression

1. Check for Multicollineariy:

In [63]:
df_clean.head()

,limit_bal,sex,education,marital_status,age_band,utilization_rate,months_late,payment_band,is_default
0,20000,0,2,1,1,19.56,2,1,1
1,120000,0,2,2,2,2.24,2,1,1
2,90000,0,2,2,2,32.49,0,2,0
3,50000,0,2,1,3,93.98,0,1,0
4,50000,1,2,1,5,17.23,0,3,0


In [ ]:
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

VIF_test = df_clean.drop('is_default', axis=1)

# Add constant (crucial for VIF Test)
X = sm.add_constant(VIF_test)

# Create table with VIF score:
df_VIF = pd.DataFrame()
df_VIF['variable'] = X.columns
df_VIF['VIF'] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]

# Print the results
df_VIF
# As seen, VIF values are all >1 but <5, which suggests that there is no multicollinearity in our variables.

# not yet finished, please check for other assumptions of logistic regression.

,variable,VIF
0,const,44.308722
1,limit_bal,1.295243
2,sex,1.022731
3,education,1.131140
4,marital_status,1.205950
5,age_band,1.245855
6,utilization_rate,1.346601
7,months_late,1.087323
8,payment_band,1.186969
